In [ ]:
import pandas as pd
import numpy as np
# goal:
#parse data from daic to the required format

import os
import shutil

diac_path = r'/Users/gurusai/Desktop/DAIC_Raw'
data_dir = r'/Users/gurusai/Desktop/review/SBT-Net/SBT-Net/depression_model/data'
if not os.path.exists(data_dir+'/audio'):
    os.makedirs(data_dir+'/audio')
if not os.path.exists(data_dir+'/transcripts'):
    os.makedirs(data_dir+'/transcripts')

dirs = [d for d in os.listdir(diac_path) 
        if os.path.isdir(os.path.join(diac_path, d))]

for file in dirs:
    if file[-2:] == "_P":
        participant = file[:-2]
        
        for doc in os.listdir(os.path.join(diac_path,file)):
            if doc[-4:] == '.wav': #obtain audio files 
                if not os.path.exists(os.path.join(data_dir,'/audio',doc)):
                    shutil.copy(os.path.join(diac_path,file,doc),data_dir+'/audio')

            if doc[4:] == "TRANSCRIPT.csv":
                if not os.path.exists(os.path.join(data_dir,'/transcripts',doc)):
                    shutil.copy(os.path.join(diac_path,file,doc),data_dir+'/transcripts')


 

In [ ]:
import torch
import pandas as pd
import os 
import numpy as np
class test():
    def __init__(self,base_dir):
          self.base_dir = base_dir

    def _load_visual_features(self, p_id):
        p_folder = os.path.join(self.base_dir, f"{p_id}_P")
        files = {
            'aus': f"{p_id}_CLNF_AUs.txt",
            'gaze': f"{p_id}_CLNF_gaze.txt",
            'pose': f"{p_id}_CLNF_pose.txt"
        }
        
        data_parts = []
        try:
            for key, filename in files.items():
                path = os.path.join(p_folder, filename)
                if not os.path.exists(path):
                    continue
                    
                df = pd.read_csv(path)
                df.columns = [c.strip() for c in df.columns]

                if key == 'aus':
                    # Grab BOTH intensity (_r) and presence (_c) to be safe
                    # This usually yields ~35 columns
                    cols = [c for c in df.columns if '_r' in c or '_c' in c]
                elif key == 'gaze':
                    # Gaze 0, Gaze 1, and Gaze Anglesß
                    cols = [c for c in df.columns if 'gaze' in c and 'angle' not in c]
                else: # pose
                    # Translation (T) and Rotation (R)
                    cols = [c for c in df.columns if 'pose_R' in c or 'pose_T' in c]

                subset = df[cols].apply(pd.to_numeric, errors='coerce').fillna(0)
                data_parts.append(subset.values)

            # Combine them horizontally
            combined = np.hstack(data_parts) 
            v_tensor = torch.from_numpy(combined).float()

            # Temporal Normalization (Target 60 frames)
            target_len = 60 
            if v_tensor.shape[0] > target_len:
                v_tensor = v_tensor[:target_len, :]
            else:
                pad_len = target_len - v_tensor.shape[0]
                v_tensor = torch.nn.functional.pad(v_tensor, (0, 0, 0, pad_len))

            return v_tensor # This should now be [60, Total_Features]

        except Exception as e:
            # Fallback to zeros if anything fails
            return torch.zeros((60, 50))

In [31]:
import os
import pandas as pd
import numpy as np

In [35]:
tensor = test(r'/Users/gurusai/Desktop/DAIC_Raw')._load_visual_features(301)

In [36]:
tensor.shape

torch.Size([60, 20])

In [9]:
_load_visual_features('/Users/gurusai/Desktop/DAIC_Raw',300)



tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [23]:
pd.read_csv('/Users/gurusai/Desktop/DAIC_Raw/300_P/300_CLNF_hog.txt')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 14: invalid start byte